In [ ]:
# ========== TFLite Micro INT8 양자화(PTQ) 변환 ==========
# 전제: Keras 모델 객체 `model` 이 메모리에 존재하거나, 저장된 SavedModel/Weights로 복구 가능
# - 입력/출력은 배치=1, 고정 길이(정적 shape) 권장
# - 모든 연산이 INT8 양자화 가능해야 MCU(TFLM+CMSIS-NN)에서 원활

import os
import numpy as np
import tensorflow as tf
from pathlib import Path

# (옵션) 필요한 경우: 저장된 모델을 불러오기 예시
# model = tf.keras.models.load_model("./saved_model_dir")

# 대표 데이터셋(Representative Dataset) 구성
# 1) tf.data.Dataset(train_ds)이 있다면 그걸 사용 (x만 꺼내도록 구성)
# 2) 없다면 X_train(numpy, float32, shape=[N, H, W, C] 또는 [N, T, C]) 사용
# 아래 함수는 둘 중 정의된 것을 자동 사용하도록 예시를 제공합니다.

def make_representative_dataset(max_samples=200):
    samples = []

    # 우선 tf.data 경로 시도
    if 'train_ds' in globals():
        try:
            for i, batch in enumerate(train_ds):
                # (x, y) 또는 (x,) 형태 모두 대응
                if isinstance(batch, (tuple, list)):
                    x = batch[0]
                else:
                    x = batch
                x = tf.cast(x, tf.float32)
                # 배치에서 1개 혹은 소량만 추출
                x = x[:1]
                samples.append(x.numpy())
                if len(samples) >= max_samples:
                    break
        except Exception:
            pass

    # numpy 경로 시도
    if len(samples) == 0 and 'X_train' in globals():
        Xn = X_train.astype('float32')
        # 샘플 서브셋
        step = max(1, len(Xn)//max_samples)
        for i in range(0, len(Xn), step):
            samples.append(Xn[i:i+1])
            if len(samples) >= max_samples:
                break

    if len(samples) == 0:
        raise RuntimeError("대표 데이터셋을 만들 수 없습니다. 'train_ds' 또는 'X_train'을 준비하세요.")

    def gen():
        for s in samples:
            # TFLite는 리스트 형태의 입력을 요구
            yield [s]
    return gen

# INT8 전량자화 변환
save_dir = Path("./export_tflm")
save_dir.mkdir(parents=True, exist_ok=True)

# Keras 모델에서 직접 변환 (SavedModel 사용 시 from_saved_model로 교체)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = make_representative_dataset(max_samples=200)()
# Builtins INT8만 사용하여 MCU 호환성 극대화
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# 동적/가변 길이 입력은 피하고, 정적 shape로 export 권장 (배치=1)
# 모델 정의 단계에서 이미 고정되어 있어야 합니다.

tflite_model = converter.convert()
with open(save_dir / "model_int8.tflite", "wb") as f:
    f.write(tflite_model)

print("[OK] Saved:", str(save_dir / "model_int8.tflite"))


In [ ]:
# ========== 0) 드라이브 마운트 ==========
from google.colab import drive
drive.mount('/content/drive')

# ========== 1) 경로/임포트 ==========
from pathlib import Path
import os, re, zipfile, pandas as pd

ROOT_DIR    = Path('/content/drive/MyDrive/IMU_DATA')   # 드라이브 내 작업 폴더
EXTRACT_DIR = Path('/content/IMU_DATA_extracted')       # Colab 런타임 로컬
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# ========== 2) ZIP 자동 탐색 & 해제 (없으면 스킵) ==========
zip_candidates = sorted(ROOT_DIR.glob('*.zip'), key=lambda p: p.stat().st_mtime, reverse=True)
if zip_candidates:
    ZIP_PATH = zip_candidates[0]
    print(f"[INFO] 사용할 ZIP: {ZIP_PATH.name}")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print(f"[OK] 압축 해제 완료 → {EXTRACT_DIR.resolve()}")
else:
    print(f"[WARN] ZIP을 찾지 못했습니다: {ROOT_DIR}  (이미 해제된 폴더를 사용합니다)")

# ========== 3) label_* 폴더의 '공통 상위 경로' 자동 탐색 ==========
def _is_label_dir(p: Path):
    name = p.name.lower()
    # label_*, class_*, good/bad 형태 지원
    return (
        name.startswith('label_') or
        name.startswith('class_') or
        name in {'good','bad','positive','negative'}
    )

def _list_label_dirs(base: Path):
    # rglob로 모든 하위 폴더 탐색
    return [d for d in base.rglob('*') if d.is_dir() and _is_label_dir(d)]

def _common_parent(paths):
    # 여러 경로의 공통 상위 경로 반환
    if not paths: return None
    common = os.path.commonpath([str(p) for p in paths])
    return Path(common)

def find_dataset_root(extracted: Path, fallback_root: Path):
    # 1) 해제 경로에서 label 디렉토리 찾기
    label_dirs = _list_label_dirs(extracted)
    if label_dirs:
        parent = _common_parent([d.parent for d in label_dirs])
        print(f"[INFO] DATASET_ROOT(auto): {parent}")
        return parent
    # 2) 드라이브 루트 하위에서 직접 찾기(이미 해제된 경우)
    label_dirs = _list_label_dirs(fallback_root)
    if label_dirs:
        parent = _common_parent([d.parent for d in label_dirs])
        print(f"[INFO] DATASET_ROOT(auto from DRIVE): {parent}")
        return parent
    raise FileNotFoundError("[ERROR] 'label_*' 또는 유사 폴더를 찾지 못했습니다.")

DATASET_ROOT = find_dataset_root(EXTRACT_DIR, ROOT_DIR)

# ========== 4) 라벨 매핑 자동화 + 안전 CSV 로딩 ==========
def _infer_label_map(root: Path):
    """root 하위의 label 디렉토리를 자동 매핑: label_0→0, label_1→1, ... / 그 외는 사전식 정렬 순서대로 0..K-1"""
    subdirs = [d for d in sorted(root.iterdir()) if d.is_dir() and _is_label_dir(d)]
    if not subdirs:
        # 한 단계 더 내려가서 재탐색
        subdirs = sorted([d for d in root.rglob('*') if d.is_dir() and _is_label_dir(d)], key=lambda p: p.as_posix())
    if not subdirs:
        raise FileNotFoundError("[ERROR] 라벨 디렉토리를 찾지 못했습니다.")
    mapping = {}
    for d in subdirs:
        name = d.name
        m = re.match(r'^(?:label|class)_(\d+)$', name, flags=re.IGNORECASE)
        if m:
            y = int(m.group(1))
        else:
            # good/bad 등은 사전식 정렬 순으로 0..K-1
            # 단, 관례상 'bad/negative'를 0, 'good/positive'를 1로 맞추려면 아래 우선순위 가중치 제공
            order_bias = {'bad':0, 'negative':0, 'good':1, 'positive':1}
            y = order_bias.get(name.lower(), None)
            if y is None:
                # 임시 None이면 나중에 정렬해서 인덱스 부여
                y = None
        mapping[name] = y
    # None이 남아있으면 사전식 정렬 순서대로 빈 ID를 채움
    used = {v for v in mapping.values() if v is not None}
    next_ids = [i for i in range(len(mapping)) if i not in used]
    for k in sorted([k for k,v in mapping.items() if v is None]):
        mapping[k] = next_ids.pop(0)
    return mapping

def _read_csv_safely(path: Path, encoding_pref=('utf-8-sig','cp949','utf-8')):
    last_err = None
    for enc in encoding_pref:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last_err = e
    # 마지막 시도로 인코딩 미지정
    try:
        return pd.read_csv(path)
    except Exception:
        raise last_err

def load_labeled_imu(root_dir: Path,
                     label_map='auto',
                     pattern=('*.csv', '*.CSV'),
                     verbose=True,
                     preview_files=5):
    root = Path(root_dir)
    if not root.exists():
        raise FileNotFoundError(f"[ERROR] 루트가 없습니다: {root}")

    if label_map == 'auto':
        label_map_used = _infer_label_map(root)
    else:
        label_map_used = dict(label_map)

    print(f"[INFO] 라벨 매핑: {label_map_used}")

    total_files, dfs = 0, []
    for subdir_name, y in label_map_used.items():
        d = root / subdir_name
        if not (d.exists() and d.is_dir()):
            print(f"[MISS] 폴더 없음: {d}")
            continue
        # 여러 패턴 지원
        files = []
        for pat in pattern:
            files += list(d.glob(pat))
        files = sorted(set(files))
        n = len(files); total_files += n
        print(f"[OK] {d.name} → {n}개 파일")
        if verbose and n>0:
            for f in files[:preview_files]:
                print(f"    • {f.name}")
            if n > preview_files:
                print(f"    • ... (총 {n}개)")
        for f in files:
            try:
                df = _read_csv_safely(f)
            except Exception as e:
                print(f"[WARN] 읽기 실패: {f.name} ({e})"); continue
            # 메타정보
            df['label']       = y
            df['source_file'] = f.name
            df['source_dir']  = subdir_name
            dfs.append(df)

    if total_files == 0 or not dfs:
        raise FileNotFoundError("[ERROR] CSV를 찾지 못했습니다. 경로/패턴을 확인하세요.")

    data = pd.concat(dfs, axis=0, ignore_index=True, sort=False)

    # 로딩 품질 점검 로그(선택)
    def _normalize_name(s):
        s = re.sub(r'[^A-Za-z0-9]+',' ', str(s)).strip().lower()
        return re.sub(r'\s+',' ', s)
    cols_norm = {_normalize_name(c): c for c in data.columns}
    move_candidates = [c for c in data.columns if _normalize_name(c) in ['ak','move','rep','segment','cycle','trial','action']]
    acc_candidates  = [c for c in data.columns if any(k in _normalize_name(c) for k in ['acc','accelerometer'])]
    gyr_candidates  = [c for c in data.columns if any(k in _normalize_name(c) for k in ['gyr','gyro'])]

    print("\n[SUMMARY]")
    print(f" - 총 CSV 파일 수: {total_files}개")
    print(f" - 총 로우 수: {len(data):,}")
    print(f" - 라벨 분포:\n{data['label'].value_counts(dropna=False).to_string()}")
    if move_candidates:
        print(f" - move 관련 열 후보: {move_candidates}")
    else:
        print(" - [WARN] move(AK) 열을 찾지 못했습니다. 전처리 전에 열 이름을 확인하세요.")
    print(f" - 가속도 열 후보: {acc_candidates[:6]}")
    print(f" - 자이로 열 후보: {gyr_candidates[:6]}")

    return data, label_map_used

# ========== 5) 실제 로딩 ==========
imu_df, label_map_used = load_labeled_imu(
    DATASET_ROOT,
    label_map='auto',            # 필요하면 {'label_0':0,'label_1':1}로 고정 가능
    pattern=('*.csv','*.CSV'),
    verbose=True,
    preview_files=5
)

# ========== 6) (옵션) 업로드 CSV 폴백 병합 ==========
# Colab 외부에서 개별 CSV를 올려둔 경우(예: ChatGPT에서 제공) 병합
fallback_csvs = [Path('/mnt/data/IMU_Label_Plus_ALL.csv'), Path('/mnt/data/IMU_label_1_jungro.csv')]
fallback_exist = [p for p in fallback_csvs if p.exists()]
if fallback_exist:
    add_dfs = []
    for p in fallback_exist:_


Mounted at /content/drive
[INFO] 사용할 ZIP: IMU_all.zip
[OK] 압축 해제 완료 → /content/IMU_DATA_extracted
[INFO] DATASET_ROOT(auto): /content/IMU_DATA_extracted
[INFO] 라벨 매핑: {'label_0': 0, 'label_1': 1}
[OK] label_0 → 1개 파일
    • IMU_Label_Plus_ALL.csv
[OK] label_1 → 1개 파일
    • IMU_label_1_jungro.csv

[SUMMARY]
 - 총 CSV 파일 수: 2개
 - 총 로우 수: 45,143
 - 라벨 분포:
label
0    36535
1     8608
 - move 관련 열 후보: ['move']
 - 가속도 열 후보: []
 - 자이로 열 후보: []


In [ ]:
# ================== MOVE 단위: Pad/Crop + Mask 전처리 (move 열 패치 포함) ==================
# - 평균 90샘플이지만 L을 더 크게(예: 128) 잡고, 짧은 건 패딩, 긴 건 크롭
# - '이벤트 중심(event-center)' 정렬: 바닥(최저점) 이벤트를 L*anchor 위치에 정렬
# - 중력정렬 옵션, 채널별 z-score(마스크 고려) 및 npz 저장
# ======================================================================

import numpy as np, pandas as pd, re, json
from pathlib import Path

# -------- Config --------
FS = 50.0
TARGET_SEC = 1.81
L = 128                          # 평균(≈90)보다 여유있게. 128~192 권장
MIN_MOVE_SEC = 0.50
PAD_ALIGN = "event"              # 'right' | 'center' | 'event'
EVENT_ANCHOR = 0.50              # 이벤트를 L*anchor에 정렬(0.5=중앙)
PAD_VALUE = "zero"               # 'zero' | 'edge' | 'mean'
GRAVITY_ALIGN = True
LPF_CUTOFF_HZ = 1.5

OUT_DIR = Path('/content/IMU_DATA_extracted'); OUT_DIR.mkdir(parents=True, exist_ok=True)
NPZ_PATH   = OUT_DIR / f'move_pad_fs{int(FS)}_L{L}_{PAD_ALIGN}_{PAD_VALUE}.npz'
SCALE_JSON = OUT_DIR / f'move_pad_fs{int(FS)}_L{L}_scaler.json'

# -------- Helpers: column map / time resample --------
def _normalize_name(s):
    s = re.sub(r'[^A-Za-z0-9]+',' ', str(s)).strip().lower()
    return re.sub(r'\s+',' ', s)

TIME_CAND = ['time','timestamp','time_ms','t','timestep','ts']
MOVE_CAND = ['ak','move','rep','segment','cycle','trial','action']

CANONICAL = {
    'ax':['ax','acc_x','accx','accelerometer x','a_x','acc-x','accX','AccX'],
    'ay':['ay','acc_y','accy','accelerometer y','a_y','acc-y','accY','AccY'],
    'az':['az','acc_z','accz','accelerometer z','a_z','acc-z','accZ','AccZ'],
    'gx':['gx','gyro_x','gyrox','gyro x','g_x','gyrX','GyroX'],
    'gy':['gy','gyro_y','gyroy','gyro y','g_y','gyrY','GyroY'],
    'gz':['gz','gyro_z','gyroz','gyro z','g_z','gyrZ','GyroZ']
}

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """채널 6개 + (있으면) time/move + 메타(label/source_*)만 남기고 표준화 이름으로 리네임."""
    cols_norm = {c:_normalize_name(c) for c in df.columns}

    # time 후보
    time_col = None
    for c,n in cols_norm.items():
        if any(n == _normalize_name(k) for k in TIME_CAND) or 'time' in n:
            time_col = c; break

    # move/AK 후보 → 'move'로 보존
    move_col = None
    for c,n in cols_norm.items():
        if any(n == _normalize_name(k) for k in MOVE_CAND) or any(_normalize_name(k) in n for k in MOVE_CAND):
            move_col = c; break

    # 6채널 매핑
    taken, mapping = set(), {}
    def find(keys):
        for c,n in cols_norm.items():
            if c in taken: continue
            if any(n == _normalize_name(k) for k in keys) or any(_normalize_name(k) in n for k in keys):
                taken.add(c); return c
        return None
    for canon, keys in CANONICAL.items():
        hit = find(keys)
        if hit is None:
            raise ValueError(f"[ERROR] 6채널 매핑 실패: {canon} 누락. 현재 컬럼 예: {list(df.columns)[:12]}")
        mapping[canon] = hit

    # 리네임 맵
    colmap = {v:k for k,v in mapping.items()}
    if time_col is not None: colmap[time_col] = 'time'
    if move_col is not None: colmap[move_col] = 'move'

    df2 = df.rename(columns=colmap)

    keep = (['time'] if 'time' in df2.columns else []) + \
           ['ax','ay','az','gx','gy','gz','label','source_file','source_dir']
    if 'move' in df2.columns: keep.append('move')

    keep = [c for c in keep if c in df2.columns]
    return df2[keep]

def resample_uniform(df_seq: pd.DataFrame, fs=FS):
    if 'time' in df_seq.columns:
        t = df_seq['time'].astype(float).to_numpy()
        if len(t)==0: return None,0
        t = t - np.nanmin(t); span = np.nanmax(t) - np.nanmin(t)
        if span > 200: t = t/1000.0     # ms -> s
        dur = float(t[-1]) if len(t)>1 else (len(df_seq)/fs)
        n_target = max(2, int(round(dur*fs)))
        t_new = np.linspace(0.0, dur, n_target)
        X = df_seq[['ax','ay','az','gx','gy','gz']].to_numpy(dtype=float)
        Xr = np.vstack([np.interp(t_new, t, X[:,i]) for i in range(X.shape[1])]).T
        return Xr, dur
    X = df_seq[['ax','ay','az','gx','gy','gz']].to_numpy(dtype=float)
    dur = len(X)/fs
    return X.copy(), dur

# -------- Gravity align (safe fallback) --------
try:
    from scipy.signal import butter, filtfilt
    def _lpf(x, fs, fc=LPF_CUTOFF_HZ):
        b,a = butter(2, fc/(fs/2), btype='low'); return filtfilt(b,a,x,axis=0)
except Exception:
    def _lpf(x, fs, fc=LPF_CUTOFF_HZ):
        k=max(1,int(round(fs/max(fc,1e-3)))); w=np.ones(k)/k
        return np.vstack([np.convolve(x[:,i], w, mode='same') for i in range(x.shape[1])]).T

def _rodrigues_R(a,b,eps=1e-8):
    a=a/np.linalg.norm(a); b=b/np.linalg.norm(b)
    v=np.cross(a,b); s=np.linalg.norm(v); c=float(np.dot(a,b))
    if s<eps:
        if c>0: return np.eye(3)
        axis = np.array([1,0,0]) if abs(a[0])<0.9 else np.array([0,1,0])
        v=np.cross(a,axis); v/=np.linalg.norm(v)
        K=np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])
        return np.eye(3)+2*K@K
    K=np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])
    return np.eye(3)+K+K@K*((1-c)/(s**2))

def gravity_align_and_linearize(X_acc, X_gyr, fs=FS):
    g = _lpf(X_acc, fs); g_mean = g.mean(axis=0)
    R = np.eye(3) if np.linalg.norm(g_mean)<1e-9 else _rodrigues_R(g_mean, np.array([0,0,1.0]))
    acc_w = (R @ X_acc.T).T; gyr_w = (R @ X_gyr.T).T
    acc_w[:,2] -= acc_w[:,2].mean()
    return acc_w, gyr_w

# -------- Move segmentation --------
def find_move_column(df, hints=('AK','move','Move','MOVE','rep','segment','cycle','trial','action')):
    cols=list(df.columns)
    for h in hints:
        if h in cols: return h
    for c in cols:
        n=_normalize_name(c)
        if any(k in n for k in ['move','segment','rep','cycle','trial','action','ak']):
            return c
    raise ValueError("move 열을 찾지 못함")

def contiguous_runs(mask_bool):
    m=np.asarray(mask_bool, dtype=bool)
    if m.size==0: return []
    diff=np.diff(np.concatenate(([False],m,[False])).astype(int))
    st=np.where(diff==1)[0]; en=np.where(diff==-1)[0]
    return list(zip(st,en))

def split_by_move(df, move_col):
    v=df[move_col]
    if v.dtype=='O' or str(v.dtype).startswith('category'):
        val=v.astype(str).str.lower().fillna('')
        return contiguous_runs(val.str.contains('move').to_numpy())
    if v.dropna().isin([0,1,True,False]).all():
        return contiguous_runs(v.astype(bool).fillna(False).to_numpy())
    arr=v.to_numpy()
    if np.isnan(arr).any():
        arr=pd.Series(arr).fillna(method='ffill').fillna(method='bfill').to_numpy()
    ch=np.r_[True, arr[1:]!=arr[:-1], True]; idx=np.flatnonzero(ch)
    return [(idx[i], idx[i+1]) for i in range(len(idx)-1)]

# -------- Event detection (bottom on acc_z) --------
def detect_bottom_index(acc_z):
    if len(acc_z)<4: return len(acc_z)//2
    return int(np.argmin(acc_z))

# -------- Pad/Crop placement --------
def place_with_mask(X, L, align='event', pad_value='zero', event_idx=None, anchor=0.5):
    """
    X: (n, C)
    반환: Xi (L,C), mask (L,)  [유효=1, 패딩=0]
    """
    n, C = X.shape
    # 긴 경우: 크롭 윈도우 시작 결정
    if n > L:
        if align == 'right':
            start = n - L
        elif align == 'center':
            start = max(0, (n - L)//2)
        elif align == 'event' and event_idx is not None:
            target = int(round(anchor * L))
            start = np.clip(event_idx - target, 0, n - L)
        else:
            start = 0
        Xsub = X[start:start+L]
        mask = np.ones(L, dtype=np.float32)
        return Xsub, mask

    # 짧은 경우: 패딩
    if pad_value == 'zero':
        canvas = np.zeros((L, C), dtype=float)
    elif pad_value == 'mean':
        meanv = X.mean(axis=0, keepdims=True)
        canvas = np.repeat(meanv, L, axis=0)
    elif pad_value == 'edge':
        canvas = np.repeat(X[-1:, :], L, axis=0)  # 끝값 반복
    else:
        canvas = np.zeros((L, C), dtype=float)

    if align == 'right':
        start = L - n
    elif align == 'center':
        start = (L - n)//2
    elif align == 'event' and event_idx is not None:
        target = int(round(anchor * L))
        start = np.clip(target - event_idx, 0, L - n)
    else:
        start = 0

    canvas[start:start+n] = X
    mask = np.zeros(L, dtype=np.float32)
    mask[start:start+n] = 1.0
    return canvas, mask

# -------- Build by move (Pad/Crop) --------
def build_by_move_pad(imu_df):
    df_std = standardize_columns(imu_df.copy())
    has_src = all(c in df_std.columns for c in ['source_dir','source_file'])
    groups = df_std.groupby(['source_dir','source_file'], sort=False) if has_src else [('all','all', df_std)]

    try:
        mv_global = find_move_column(imu_df)
    except: mv_global=None

    Xs, Ms, ys, durs, srcs = [], [], [], [], []
    it = ((d,f,g) for (d,f),g in groups) if not isinstance(groups, list) else [('all','all', df_std)]
    dropped = 0

    for dname, fname, g in it:
        g = g.reset_index(drop=True)
        mv_col = mv_global if (mv_global and mv_global in g.columns) else None
        if mv_col is None:
            try: mv_col = find_move_column(g)
            except:
                print(f"[WARN] move 열 없음: {dname}/{fname}"); continue

        segs = split_by_move(g, mv_col)
        if not segs:
            print(f"[WARN] move 구간 없음: {dname}/{fname}"); continue

        for s,e in segs:
            seg = g.iloc[s:e]
            if len(seg) < int(MIN_MOVE_SEC*FS):
                dropped += 1; continue

            Xr, dur = resample_uniform(seg, FS)
            if Xr is None or len(Xr)==0: continue

            if GRAVITY_ALIGN:
                acc_w, gyr_w = gravity_align_and_linearize(Xr[:,:3], Xr[:,3:], FS)
                Xr = np.concatenate([acc_w, gyr_w], axis=1)

            event_idx = detect_bottom_index(Xr[:,2]) if PAD_ALIGN=='event' else None
            Xi, Mi = place_with_mask(Xr, L, align=PAD_ALIGN, pad_value=PAD_VALUE,
                                     event_idx=event_idx, anchor=EVENT_ANCHOR)

            if not np.all(np.isfinite(Xi)):
                Xi = np.nan_to_num(Xi, nan=0.0, posinf=0.0, neginf=0.0)

            y = int(seg['label'].mode().iloc[0]) if 'label' in seg.columns else 0
            Xs.append(Xi); Ms.append(Mi); ys.append(y); durs.append(dur)
            srcs.append(f"{dname}/{fname}#move[{s}:{e}]")

    if len(Xs)==0: raise RuntimeError("생성된 move가 없습니다. (move/AK 열을 확인하세요)")
    X = np.stack(Xs, axis=0)             # (N,L,6)
    M = np.stack(Ms, axis=0)             # (N,L)
    y = np.array(ys, dtype=int)
    durs = np.array(durs, dtype=float)

    print("\n[MOVE PAD SUMMARY]")
    print(" - N:", len(X))
    print(" - X:", X.shape, "(N,L,6)")
    print(" - M:", M.shape, "(N,L)  (1=valid, 0=pad)")
    print(" - y dist:\n", pd.Series(y).value_counts(dropna=False).to_string())
    print(" - dur mean/p50/p90/min/max (s): %.3f / %.3f / %.3f / %.3f / %.3f" %
          (durs.mean(), np.percentile(durs,50), np.percentile(durs,90), durs.min(), durs.max()))
    return X, M, y, durs, srcs

# -------- Split & Masked Z-score --------
from sklearn.model_selection import StratifiedShuffleSplit

def safe_split_idx(y, seed=42, test_size=0.2, val_size=0.5):
    uniq, cnt = np.unique(y, return_counts=True)
    if len(uniq)<2: raise RuntimeError("최소 2개 클래스 필요")
    if (cnt<2).any() or len(y)<5:
        print("[WARN] 적은 샘플 → 비계층 분할")
        idx=np.arange(len(y)); rng=np.random.RandomState(seed); rng.shuffle(idx)
        n_test=max(1,int(round(test_size*len(y)))); te=idx[:n_test]; rem=idx[n_test:]
        n_val=max(1,int(round(val_size*len(rem)))); va=rem[:n_val]; tr=rem[n_val:]
        return tr, va, te
    sss1=StratifiedShuffleSplit(n_splits=1,test_size=test_size,random_state=seed)
    tr, te = next(sss1.split(np.zeros(len(y)), y))
    y_tmp = y[te]
    sss2=StratifiedShuffleSplit(n_splits=1,test_size=val_size,random_state=seed)
    va_rel, te_rel = next(sss2.split(np.zeros(len(y_tmp)), y_tmp))
    va, te = te[va_rel], te[te_rel]
    return tr, va, te

def fit_masked_zscore(X, M):
    # X:(N,L,C), M:(N,L)
    w = M[..., None]                          # (N,L,1)
    count = w.sum(axis=(0,1))                 # (C,)가 아님 주의! 아래에서 브로드캐스트로 사용
    count[count==0]=1.0
    mu = (X*w).sum(axis=(0,1))/count          # (C,) 로 브로드캐스트됨
    var= ((X - mu)**2 * w).sum(axis=(0,1))/count
    sigma = np.sqrt(np.maximum(var, 1e-12))
    return mu, sigma

def apply_zscore_masked(X, mu, sigma):
    return (X - mu) / sigma

# -------- Load imu_df (fallback) --------
def _fallback_load():
    cands=[Path('/mnt/data/IMU_Label_Plus_ALL.csv'), Path('/mnt/data/IMU_label_1_jungro.csv')]
    dfs=[]
    for p in cands:
        if p.exists():
            try: dfs.append(pd.read_csv(p)); print("[INFO] read", p)
            except Exception as e: print("[WARN] read fail", p, e)
    if dfs:
        df=pd.concat(dfs, ignore_index=True)
        if 'source_file' not in df.columns: df['source_file']='uploaded.csv'
        if 'source_dir'  not in df.columns: df['source_dir']='label_?'
        return df
    root=Path('/content/drive/MyDrive/IMU_DATA'); merged=root/'_merged_all_with_labels.csv'
    if merged.exists(): print("[INFO] read", merged); return pd.read_csv(merged)
    raise RuntimeError("imu_df 없음 & fallback 없음")

# -------- Run --------
try:
    imu_df  # 이미 존재하면 사용
except NameError:
    imu_df = _fallback_load()

X, M, y, durs, srcs = build_by_move_pad(imu_df)
tr, va, te = safe_split_idx(y, seed=42, test_size=0.2, val_size=0.5)

Xtr, ytr, Mtr = X[tr], y[tr], M[tr]
Xva, yva, Mva = X[va], y[va], M[va]
Xte, yte, Mte = X[te], y[te], M[te]

# Masked z-score (train 기준, pad 무시)
mu, sigma = fit_masked_zscore(Xtr, Mtr)
Xtr_n = apply_zscore_masked(Xtr, mu, sigma)
Xva_n = apply_zscore_masked(Xva, mu, sigma)
Xte_n = apply_zscore_masked(Xte, mu, sigma)

# Save
np.savez_compressed(NPZ_PATH,
    X_train=Xtr_n, y_train=ytr, mask_train=Mtr,
    X_val=Xva_n,   y_val=yva,   mask_val=Mva,
    X_test=Xte_n,  y_test=yte,  mask_test=Mte)

with open(SCALE_JSON, 'w') as f:
    json.dump({'mu':mu.tolist(),'sigma':sigma.tolist(),
               'fs':FS,'L':L,'pad_align':PAD_ALIGN,'pad_value':PAD_VALUE,
               'gravity_align':GRAVITY_ALIGN}, f, indent=2)

print(f"\n[OK] saved → {NPZ_PATH}")
print(f"[OK] scaler → {SCALE_JSON}")
# ================== END ==================


RuntimeError: imu_df 없음 & fallback 없음

In [ ]:
# ===================== Improved TCN Training (OOM-safe, fixes applied) =====================
import os, glob, json, numpy as np
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

# -------------------- (Optional) GPU 메모리 설정 & 혼합정밀도 --------------------
# GPU 메모리 점유 급증 방지: 필요 시 메모리 growth
try:
    gpus = tf.config.experimental.list_physical_devices('GPU')
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
except Exception as e:
    print("[WARN] set_memory_growth skipped:", e)

# 혼합정밀도(메모리 절감) 활성화
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')  # 약 30~40% 메모리 절감 기대

# -------------------- Load preprocessed NPZ --------------------
OUT_DIR = Path('/content/IMU_DATA_extracted')
cands = sorted(glob.glob(str(OUT_DIR / 'move_pad_*.npz')), key=os.path.getmtime, reverse=True)
if not cands:
    raise FileNotFoundError("전처리 NPZ가 없습니다. 먼저 전처리 파이프라인을 실행하세요.")
NPZ_PATH = cands[0]
print(f"[INFO] NPZ: {Path(NPZ_PATH).name}")

npz = np.load(NPZ_PATH)
Xtr, Xva, Xte = npz['X_train'], npz['X_val'], npz['X_test']       # (N,L,C)
ytr, yva, yte = npz['y_train'], npz['y_val'], npz['y_test']       # (N,)
Mtr, Mva, Mte = npz['mask_train'], npz['mask_val'], npz['mask_test']  # (N,L)
Mtr, Mva, Mte = Mtr[...,None], Mva[...,None], Mte[...,None]       # (N,L,1)

Ntr, L, C0 = Xtr.shape
num_classes = int(max(ytr.max(), yva.max(), yte.max()) + 1)
assert num_classes >= 2, "이 코드는 2개 이상의 클래스에서 동작합니다."
print(f"[INFO] Shapes -> Xtr:{Xtr.shape}, Xva:{Xva.shape}, Xte:{Xte.shape}, classes={num_classes}")

# -------------------- Feature enhancement: add L2 norms (acc, gyro) --------------------
def add_feature_channels(X):
    # X:(N,L,C0) with channel order [ax,ay,az,gx,gy,gz,...]
    ax, ay, az = X[:,:,0], X[:,:,1], X[:,:,2]
    gx, gy, gz = X[:,:,3], X[:,:,4], X[:,:,5]
    acc_norm = np.sqrt(ax*ax + ay*ay + az*az)[...,None]
    gyr_norm = np.sqrt(gx*gx + gy*gy + gz*gz)[...,None]
    return np.concatenate([X, acc_norm, gyr_norm], axis=-1)

Xtr = add_feature_channels(Xtr)
Xva = add_feature_channels(Xva)
Xte = add_feature_channels(Xte)
C = Xtr.shape[-1]
print(f"[INFO] Feature-augmented channels: {C} (was {C0}, added 2 norms)")

# -------------------- Oversampling minority class on TRAIN (max_ratio 제한) --------------------
def oversample_minority(X, M, y, seed=42, max_ratio=2.0):
    classes, counts = np.unique(y, return_counts=True)
    minc = classes[np.argmin(counts)]
    nmin = counts.min()
    target = int(min(counts.max(), nmin * max_ratio))  # 소수클래스 최대 2배까지만 증폭
    if nmin >= target:
        return X, M, y
    rng = np.random.RandomState(seed)
    min_idx = np.where(y==minc)[0]
    extra = rng.choice(min_idx, size=target - nmin, replace=True)
    idx = np.r_[np.arange(len(y)), extra]
    rng.shuffle(idx)
    return X[idx], M[idx], y[idx]

Xtr_os, Mtr_os, ytr_os = oversample_minority(Xtr, Mtr, ytr, max_ratio=2.0)
print(f"[INFO] Oversampled train: {Xtr.shape[0]} -> {Xtr_os.shape[0]} (class dist: {np.bincount(ytr_os)})")

# -------------------- Data augmentation (train only) --------------------
# yaw z-rotation (-15°~+15°), time-shift (-4~+4), Gaussian noise (std=0.02), channel dropout (p=0.1)
AUG_CFG = dict(yaw_deg=15.0, max_shift=4, noise_std=0.02, ch_drop_p=0.10)

def augment_np(x, m):
    # x:(L,C), m:(L,1)
    L_, C_ = x.shape
    # 1) yaw rotation on acc(x,y) & gyro(x,y)
    th = np.deg2rad(np.random.uniform(-AUG_CFG['yaw_deg'], AUG_CFG['yaw_deg']))
    c, s = np.cos(th), np.sin(th)
    xxy = x[:, [0,1]].copy(); gxy = x[:, [3,4]].copy()  # (ax,ay), (gx,gy)
    x[:,0] = c*xxy[:,0] - s*xxy[:,1]
    x[:,1] = s*xxy[:,0] + c*xxy[:,1]
    x[:,3] = c*gxy[:,0] - s*gxy[:,1]
    x[:,4] = s*gxy[:,0] + c*gxy[:,1]
    # 2) small time shift
    sh = np.random.randint(-AUG_CFG['max_shift'], AUG_CFG['max_shift']+1)
    if sh != 0:
        x = np.roll(x, sh, axis=0)
        m = np.roll(m, sh, axis=0)
    # 3) gaussian noise
    x = x + np.random.normal(0.0, AUG_CFG['noise_std'], size=x.shape).astype(np.float32)
    # 4) channel drop along time
    if np.random.rand() < AUG_CFG['ch_drop_p']:
        t0 = np.random.randint(0, max(1, L_-8))
        t1 = min(L_, t0 + np.random.randint(4, 12))
        x[t0:t1, :] *= 0.0
        m[t0:t1, :] *= 0.0
    return x.astype(np.float32), m.astype(np.float32)

def tf_augment(x, m, y):
    x, m = tf.numpy_function(func=augment_np, inp=[x, m], Tout=[tf.float32, tf.float32])
    x.set_shape([L, C]); m.set_shape([L, 1])
    return (x, m), y

# -------------------- TF datasets (메모리 완화: batch↓, prefetch(1), shuffle buffer 제한, 병렬=1) --------------------
BATCH = 32  # 64 -> 32로 축소 (필요 시 16)
def make_ds(X, M, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X.astype('float32'),
                                             M.astype('float32'),
                                             y.astype('int32')))
    if training:
        buf = int(min(len(y), 2048))  # 대형 버퍼 방지
        ds = ds.shuffle(buffer_size=buf, seed=42, reshuffle_each_iteration=True)
        ds = ds.map(tf_augment, num_parallel_calls=1, deterministic=True)
    else:
        ds = ds.map(lambda x, m, y: ((x, m), y), num_parallel_calls=1, deterministic=True)
    ds = ds.batch(BATCH, drop_remainder=False)
    ds = ds.prefetch(1)  # AUTOTUNE -> 1 (피크 억제)
    return ds

ds_tr = make_ds(Xtr_os, Mtr_os, ytr_os, training=True)
ds_va = make_ds(Xva,    Mva,    yva,    training=False)
ds_te = make_ds(Xte,    Mte,    yte,    training=False)

# -------------------- Model: stronger TCN (LayerNorm) + 경량 옵션 --------------------
def tcn_block(x, filters, k=5, d=1, pdrop=0.2, name='tcn'):
    y = layers.Conv1D(filters, k, padding='same', dilation_rate=d, name=f'{name}_conv1')(x)
    y = layers.LayerNormalization(name=f'{name}_ln1')(y)
    y = layers.Activation('relu', name=f'{name}_relu1')(y)
    y = layers.SpatialDropout1D(pdrop, name=f'{name}_sd1')(y)
    y = layers.Conv1D(filters, k, padding='same', dilation_rate=d, name=f'{name}_conv2')(y)
    y = layers.LayerNormalization(name=f'{name}_ln2')(y)
    # Residual projection if needed
    if x.shape[-1] != filters:
        x = layers.Conv1D(filters, 1, padding='same', name=f'{name}_proj')(x)
    out = layers.Add(name=f'{name}_add')([x, y])
    out = layers.Activation('relu', name=f'{name}_relu2')(out)
    return out

x_in = layers.Input(shape=(L, C), name='x')
m_in = layers.Input(shape=(L, 1), name='mask')

# 마스크 적용
x = layers.Multiply(name='apply_mask')([x_in, m_in])

# 경량화: filters 64 -> 48, dilations 단계 유지(원하면 줄여도 됨: [1,2,4,8])
filters = 48
k = 5
pdrop = 0.2
dils = [1,2,4,8,16]

for i, d in enumerate(dils):
    x = tcn_block(x, filters=filters, k=k, d=d, pdrop=pdrop, name=f'tcn{i+1}')

# Masked GAP (혼합정밀도에서도 안전하게 FP32 누적)
def masked_gap(x_and_m):
    x, m = x_and_m
    x = tf.cast(x, tf.float32)
    m = tf.cast(m, tf.float32)
    sum_x = tf.reduce_sum(x * m, axis=1)         # (B, C)
    sum_m = tf.reduce_sum(m, axis=1) + 1e-6      # (B, 1)
    return sum_x / sum_m                          # FP32 반환

gap = layers.Lambda(masked_gap, name='masked_gap')([x, m_in])

h = layers.Dense(96, activation='relu', name='head_fc')(gap)  # 이 Dense는 정책상 FP16 compute
h = layers.Dropout(0.3)(h)
# 혼합정밀도에서 손실 안정성 위해 최종 출력만 FP32로 강제
out = layers.Dense(num_classes, activation='softmax', dtype='float32', name='pred')(h)

model = keras.Model(inputs=[x_in, m_in], outputs=out)
model.summary()

# -------------------- Focal Loss (gamma=2, alpha tuned for minority) --------------------
classes, counts = np.unique(ytr, return_counts=True)
minor = classes[np.argmin(counts)]
alpha_vec = np.ones((num_classes,), dtype=np.float32) * 0.25
alpha_vec[minor] = 0.75        # minority upweight
print(f"[INFO] focal alpha: {alpha_vec}, gamma=2.0")

def focal_loss(gamma=2.0, alpha=alpha_vec):
    sce = keras.losses.SparseCategoricalCrossentropy(reduction='none')
    alpha_c = tf.constant(alpha, dtype=tf.float32)
    def loss(y_true, y_pred):
        # clamp for numerical stability
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0-1e-7)
        y_true = tf.cast(y_true, tf.int32)
        ce = sce(y_true, y_pred)                     # (B,)
        y_oh = tf.one_hot(y_true, depth=num_classes, dtype=tf.float32)
        p_t = tf.reduce_sum(y_oh * y_pred, axis=-1)  # (B,)
        a_t = tf.reduce_sum(y_oh * alpha_c, axis=-1) # (B,)
        fl = a_t * tf.pow(1.0 - p_t, gamma) * ce
        return tf.reduce_mean(fl)
    return loss

# (선택) class_weight 병행
USE_CLASS_WEIGHT = True
class_weight = None
if USE_CLASS_WEIGHT:
    cw = compute_class_weight(class_weight='balanced', classes=np.arange(num_classes), y=ytr_os)
    class_weight = {int(i): float(w) for i, w in enumerate(cw)}
    print("[INFO] class_weight:", class_weight)

# -------------------- Macro-F1 on VAL for EarlyStopping --------------------
class ValF1Callback(keras.callbacks.Callback):
    def __init__(self, ds_val, name='val_f1_macro', patience=8):
        super().__init__(); self.ds_val = ds_val; self.name=name
        self.best = -1.0; self.patience = patience; self.wait=0
    def on_epoch_end(self, epoch, logs=None):
        probs, y_true = [], []
        for (xb, mb), yb in self.ds_val:
            pb = self.model.predict({'x': xb, 'mask': mb}, verbose=0)
            probs.append(pb); y_true.append(yb.numpy())
        probs = np.concatenate(probs, axis=0)
        y_true = np.concatenate(y_true, axis=0)
        y_pred = probs.argmax(axis=1)
        f1m = f1_score(y_true, y_pred, average='macro', zero_division=0)
        logs = logs or {}
        logs[self.name] = f1m
        print(f"\n[VAL] macro-F1={f1m:.4f}")
        if f1m > self.best + 1e-6:
            self.best = f1m; self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"[EarlyStopping] No improvement in {self.patience} epochs on {self.name}.")
                self.model.stop_training = True

# -------------------- Compile & Train (LR 스케줄러 시그니처 수정) --------------------
INIT_LR = 2.5e-3
EPOCHS  = 60
MAX_EPOCHS = EPOCHS

def cosine_decay(epoch, lr):
    import math
    t = min(epoch, MAX_EPOCHS)
    return 0.5 * INIT_LR * (1.0 + math.cos(math.pi * t / MAX_EPOCHS))

model.compile(optimizer=keras.optimizers.Adam(learning_rate=INIT_LR),
              loss=focal_loss(gamma=2.0, alpha=alpha_vec),
              metrics=['accuracy'])

valf1_cb = ValF1Callback(ds_va, name='val_f1_macro', patience=8)
lrs_cb   = keras.callbacks.LearningRateScheduler(cosine_decay, verbose=0)
ckpt_cb  = keras.callbacks.ModelCheckpoint(str(OUT_DIR / 'tcn_best.keras'),
                                           monitor='val_accuracy', save_best_only=True)

hist = model.fit(ds_tr,
                 validation_data=ds_va,
                 epochs=EPOCHS,
                 callbacks=[valf1_cb, lrs_cb, ckpt_cb],
                 class_weight=class_weight,
                 verbose=1)

# -------------------- Threshold tuning on VAL (binary only) --------------------
use_threshold = False
best_t = 0.5
if num_classes == 2:
    probs_va = []
    for (xb, mb), yb in ds_va:
        probs_va.append(model.predict({'x': xb, 'mask': mb}, verbose=0))
    probs_va = np.concatenate(probs_va, axis=0)
    p1_va = probs_va[:,1]
    best_f1 = -1
    for t in np.linspace(0.2, 0.8, 121):
        y_pred = (p1_va >= t).astype(int)
        f1m = f1_score(yva, y_pred, average='macro', zero_division=0)
        if f1m > best_f1:
            best_f1, best_t = f1m, t
    use_threshold = True
    print(f"[VAL] best threshold={best_t:.3f} | macro-F1={best_f1:.4f}")

# -------------------- Evaluation helpers --------------------
def eval_split(name, ds, y_true_full, use_thr=False, thr=0.5):
    probs = []
    for (xb, mb), yb in ds:
        probs.append(model.predict({'x': xb, 'mask': mb}, verbose=0))
    probs = np.concatenate(probs, axis=0)
    if (num_classes == 2) and use_thr:
        y_pred = (probs[:,1] >= thr).astype(int)
    else:
        y_pred = probs.argmax(axis=1)
    acc = (y_pred == y_true_full).mean()
    f1m = f1_score(y_true_full, y_pred, average='macro', zero_division=0)
    f1w = f1_score(y_true_full, y_pred, average='weighted', zero_division=0)
    prec= precision_score(y_true_full, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true_full, y_pred, average='macro', zero_division=0)
    print(f"\n[{name}] acc={acc:.4f} | F1(macro)={f1m:.4f} | F1(weighted)={f1w:.4f} | Precision={prec:.4f} | Recall={rec:.4f}")
    print("Classification report:")
    print(classification_report(y_true_full, y_pred, digits=4, zero_division=0))
    print("Confusion matrix:")
    print(confusion_matrix(y_true_full, y_pred))
    return dict(acc=acc, f1_macro=f1m, f1_weighted=f1w)

print("\n================ Final Evaluation ================")
_ = eval_split('VAL',  ds_va, yva, use_thr=use_threshold, thr=best_t)
_ = eval_split('TEST', ds_te, yte, use_thr=use_threshold, thr=best_t)

print(f"\n[OK] Best model checkpoint: {OUT_DIR/'tcn_best.keras'}")
# =======================================================================================


[INFO] NPZ: move_pad_fs50_L128_event_zero.npz
[INFO] Shapes -> Xtr:(616, 128, 6), Xva:(77, 128, 6), Xte:(78, 128, 6), classes=2
[INFO] Feature-augmented channels: 8 (was 6, added 2 norms)
[INFO] Oversampled train: 616 -> 736 (class dist: [496 240])


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ x (InputLayer)      │ (None, 128, 8)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mask (InputLayer)   │ (None, 128, 1)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ apply_mask          │ (None, 128, 8)    │          0 │ x[0][0],          │
│ (Multiply)          │                   │            │ mask[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_conv1 (Conv1D) │ (None, 128, 48)   │      1,968 │ apply_mask[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_ln1            │ (None, 128, 48)   │         96 │ tcn1_conv1[0][0]  │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_relu1          │ (None, 128, 48)   │          0 │ tcn1_ln1[0][0]    │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_sd1            │ (None, 128, 48)   │          0 │ tcn1_relu1[0][0]  │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_conv2 (Conv1D) │ (None, 128, 48)   │     11,568 │ tcn1_sd1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_proj (Conv1D)  │ (None, 128, 48)   │        432 │ apply_mask[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_ln2            │ (None, 128, 48)   │         96 │ tcn1_conv2[0][0]  │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_add (Add)      │ (None, 128, 48)   │          0 │ tcn1_proj[0][0],  │
│                     │                   │            │ tcn1_ln2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_relu2          │ (None, 128, 48)   │          0 │ tcn1_add[0][0]    │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_conv1 (Conv1D) │ (None, 128, 48)   │     11,568 │ tcn1_relu2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_ln1            │ (None, 128, 48)   │         96 │ tcn2_conv1[0][0]  │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_relu1          │ (None, 128, 48)   │          0 │ tcn2_ln1[0][0]    │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_sd1            │ (None, 128, 48)   │          0 │ tcn2_relu1[0][0]  │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_conv2 (Conv1D) │ (None, 128, 48)   │     11,568 │ tcn2_sd1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_ln2            │ (None, 128, 48)   │         96 │ tcn2_conv2[0][0]  │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_add (Add)      │ (None, 128, 48)   │          0 │ tcn1_relu2[0][0], │
│                     │                   │            │ tcn2_ln2[0][0]  

 Total params: 112,370 (438.95 KB)

 Trainable params: 112,370 (438.95 KB)

 Non-trainable params: 0 (0.00 B)

[INFO] focal alpha: [0.25 0.75], gamma=2.0
[INFO] class_weight: {0: 0.7419354838709677, 1: 1.5333333333333334}
Epoch 1/60
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5039 - loss: 0.2711
[VAL] macro-F1=0.2311
23/23 ━━━━━━━━━━━━━━━━━━━━ 38s 437ms/step - accuracy: 0.5031 - loss: 0.2676 - val_accuracy: 0.2468 - val_loss: 0.0866 - val_f1_macro: 0.2311 - learning_rate: 0.0025
Epoch 2/60
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5668 - loss: 0.0853
[VAL] macro-F1=0.5749
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - accuracy: 0.5666 - loss: 0.0849 - val_accuracy: 0.6234 - val_loss: 0.0523 - val_f1_macro: 0.5749 - learning_rate: 0.0025
Epoch 3/60
22/23 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5693 - loss: 0.0706
[VAL] macro-F1=0.6259
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.5698 - loss: 0.0702 - val_accuracy: 0.6753 - val_loss: 0.0489 - val_f1_macro: 0.6259 - learning_rate: 0.0025
Epoch 4/60
20/23 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6234 - loss:

In [ ]:
# ===== LOCAL 다운로드 (Google Colab) =====
from pathlib import Path
import zipfile

# 1) 단일 파일 다운로드: .h5
try:
    from google.colab import files
    h5_path = OUT_DIR / 'tcn_export_fp32.h5'
    assert h5_path.exists(), f"파일이 없음: {h5_path}"
    files.download(str(h5_path))
    print("[OK] .h5 다운로드를 시작했습니다.")
except Exception as e:
    print("[WARN] 단일 다운로드 실패:", e)

# 2) (옵션) 여러 파일을 한 번에 ZIP으로 묶어 다운로드
#    필요에 따라 포함 목록을 조정하세요.
bundle_list = [
    OUT_DIR / 'tcn_export_fp32.h5',
    OUT_DIR / 'tcn_best.keras',            # 베스트 체크포인트(있으면)
    OUT_DIR / 'move_pad_fs50_L192_scaler.json',  # 스케일러 JSON(경로/이름 다르면 바꾸세요)
    OUT_DIR / 'move_pad_fs50_L192_event_zero.npz'  # 전처리 NPZ(이름 다르면 바꾸세요)
]
bundle_list = [p for p in bundle_list if Path(p).exists()]

if bundle_list:
    zip_path = OUT_DIR / 'tcn_export_bundle.zip'
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for p in bundle_list:
            zf.write(p, arcname=p.name)
    try:
        from google.colab import files
        files.download(str(zip_path))
        print("[OK] 번들 ZIP 다운로드를 시작했습니다 ->", zip_path)
    except Exception as e:
        print("[WARN] ZIP 다운로드 실패:", e)

# 3) (옵션) TFLite가 있다면 같이 다운로드
#    위에서 TFLite를 만든 경우에만 사용하세요.
for tfl in ['tcn_export_fp32.tflite', 'tcn_export_int8.tflite']:
    p = OUT_DIR / tfl
    if p.exists():
        try:
            from google.colab import files
            files.download(str(p))
            print(f"[OK] {tfl} 다운로드를 시작했습니다.")
        except Exception as e:
            print(f"[WARN] {tfl} 다운로드 실패:", e)


[WARN] 단일 다운로드 실패: 파일이 없음: /content/IMU_DATA_extracted/tcn_export_fp32.h5


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[OK] 번들 ZIP 다운로드를 시작했습니다 -> /content/IMU_DATA_extracted/tcn_export_bundle.zip


In [ ]:
# ===================== TCN Training for STM32Cube.AI (single-input, no mask/Lambda) =====================
import os, glob, json, numpy as np
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

# -------------------- GPU & Mixed Precision (OOM-safe) --------------------
try:
    gpus = tf.config.experimental.list_physical_devices('GPU')
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
except Exception as e:
    print("[WARN] set_memory_growth skipped:", e)

from tensorflow.keras import mixed_precision
USE_MIXED_PRECISION = True
if USE_MIXED_PRECISION:
    mixed_precision.set_global_policy('mixed_float16')   # 학습 메모리 절감
else:
    mixed_precision.set_global_policy('float32')

# -------------------- Load preprocessed NPZ --------------------
OUT_DIR = Path('/content/IMU_DATA_extracted')
cands = sorted(glob.glob(str(OUT_DIR / 'move_pad_*.npz')), key=os.path.getmtime, reverse=True)
if not cands:
    raise FileNotFoundError("전처리 NPZ가 없습니다. 먼저 전처리 파이프라인을 실행하세요.")
NPZ_PATH = cands[0]
print(f"[INFO] NPZ: {Path(NPZ_PATH).name}")

npz = np.load(NPZ_PATH)
Xtr, Xva, Xte = npz['X_train'], npz['X_val'], npz['X_test']       # (N,L,6) 기본 가정
ytr, yva, yte = npz['y_train'], npz['y_val'], npz['y_test']       # (N,)
# mask_*는 학습 그래프에서 사용하지 않음 (단일 입력)
L = Xtr.shape[1]
C0 = Xtr.shape[2]
num_classes = int(max(ytr.max(), yva.max(), yte.max()) + 1)
assert num_classes >= 2, "이 코드는 2개 이상의 클래스에서 동작합니다."
print(f"[INFO] Shapes -> Xtr:{Xtr.shape}, Xva:{Xva.shape}, Xte:{Xte.shape}, classes={num_classes}")

# -------------------- (옵션) MCU 친화적 피처 추가: energy(제곱합) 채널 --------------------
# sqrt를 쓰지 않고 ax^2+ay^2+az^2, gx^2+gy^2+gz^2 을 채널로 추가 (장치 전처리도 덧셈/곱셈만으로 일치 가능)
ADD_ENERGY_CHANNELS = True

def add_energy_channels(X):
    # X:(N,L,C0) with [ax,ay,az,gx,gy,gz,...]
    ax, ay, az = X[...,0], X[...,1], X[...,2]
    gx, gy, gz = X[...,3], X[...,4], X[...,5]
    acc_energy = (ax*ax + ay*ay + az*az)[...,None].astype(np.float32)
    gyr_energy = (gx*gx + gy*gy + gz*gz)[...,None].astype(np.float32)
    return np.concatenate([X, acc_energy, gyr_energy], axis=-1)

if ADD_ENERGY_CHANNELS:
    Xtr = add_energy_channels(Xtr)
    Xva = add_energy_channels(Xva)
    Xte = add_energy_channels(Xte)

C = Xtr.shape[-1]
print(f"[INFO] Channels: base={C0} -> used={C} (energy_added={ADD_ENERGY_CHANNELS})")

# -------------------- Class rebalancing (train only) --------------------
def oversample_minority(X, y, seed=42, max_ratio=2.0):
    classes, counts = np.unique(y, return_counts=True)
    minc = classes[np.argmin(counts)]
    nmin = counts.min()
    target = int(min(counts.max(), nmin * max_ratio))  # 소수클래스 최대 2배
    if nmin >= target:
        return X, y
    rng = np.random.RandomState(seed)
    min_idx = np.where(y==minc)[0]
    extra = rng.choice(min_idx, size=target - nmin, replace=True)
    idx = np.r_[np.arange(len(y)), extra]
    rng.shuffle(idx)
    return X[idx], y[idx]

Xtr_os, ytr_os = oversample_minority(Xtr, ytr, max_ratio=2.0)
print(f"[INFO] Oversampled train: {Xtr.shape[0]} -> {Xtr_os.shape[0]} (class dist: {np.bincount(ytr_os)})")

# -------------------- Data augmentation (mask 없이 동작) --------------------
# yaw z-rotation (-15~+15 deg), small time shift (-4~+4), Gaussian noise (std=0.02), random zero segment (p=0.1)
AUG = dict(yaw_deg=15.0, max_shift=4, noise_std=0.02, drop_p=0.10)

def augment_np(x):
    # x: (L,C)
    x = x.copy()
    # 1) yaw 회전: acc(x,y)=0,1 / gyro(x,y)=3,4 에만 적용 (추가 채널은 그대로)
    th = np.deg2rad(np.random.uniform(-AUG['yaw_deg'], AUG['yaw_deg']))
    c, s = np.cos(th), np.sin(th)
    xxy = x[:, [0,1]].copy(); gxy = x[:, [3,4]].copy()
    x[:,0] = c*xxy[:,0] - s*xxy[:,1]
    x[:,1] = s*xxy[:,0] + c*xxy[:,1]
    x[:,3] = c*gxy[:,0] - s*gxy[:,1]
    x[:,4] = s*gxy[:,0] + c*gxy[:,1]
    # 2) small time shift (패딩이 섞일 수 있으므로 작은 범위만)
    sh = np.random.randint(-AUG['max_shift'], AUG['max_shift']+1)
    if sh != 0:
        x = np.roll(x, sh, axis=0)
    # 3) gaussian noise
    x = x + np.random.normal(0.0, AUG['noise_std'], size=x.shape).astype(np.float32)
    # 4) short zero segment
    if np.random.rand() < AUG['drop_p']:
        t0 = np.random.randint(0, max(1, L-8))
        t1 = min(L, t0 + np.random.randint(4, 12))
        x[t0:t1, :] = 0.0
    return x.astype(np.float32)

def tf_augment(x, y):
    x = tf.numpy_function(func=augment_np, inp=[x], Tout=tf.float32)
    x.set_shape([L, C])
    return x, y

# -------------------- TF datasets (OOM-safe) --------------------
BATCH = 32
def make_ds(X, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X.astype('float32'), y.astype('int32')))
    if training:
        buf = int(min(len(y), 2048))
        ds = ds.shuffle(buffer_size=buf, seed=42, reshuffle_each_iteration=True)
        ds = ds.map(tf_augment, num_parallel_calls=1, deterministic=True)
    ds = ds.batch(BATCH, drop_remainder=False).prefetch(1)
    return ds

ds_tr = make_ds(Xtr_os, ytr_os, training=True)
ds_va = make_ds(Xva,    yva,    training=False)
ds_te = make_ds(Xte,    yte,    training=False)

# -------------------- Model (Cube.AI friendly TCN) --------------------
# Conv1D(use_bias=False) + BatchNorm + ReLU (첫 블록 BN은 center=False로 0-패딩 보존),
# 잔차 연결, GlobalMaxPooling1D, Dense → Dense(softmax, dtype=float32)
FILTERS   = 48
KERNEL    = 5
DILS      = [1,2,4,8,16]     # 필요시 줄여도 됨
HEAD_UNITS= 96

def tcn_block_bn(x, filters, k=5, d=1, name='tcn', first_block=False):
    y = layers.Conv1D(filters, k, padding='same', dilation_rate=d, use_bias=False, name=f'{name}_c1')(x)
    # 첫 블록은 center=False로 0 입력이 BN에서 +β로 밀리지 않게
    y = layers.BatchNormalization(center=(not first_block), scale=True, name=f'{name}_bn1')(y)
    y = layers.ReLU(name=f'{name}_relu1')(y)
    y = layers.Conv1D(filters, k, padding='same', dilation_rate=d, use_bias=False, name=f'{name}_c2')(y)
    y = layers.BatchNormalization(name=f'{name}_bn2')(y)
    if x.shape[-1] != filters:
        x = layers.Conv1D(filters, 1, padding='same', use_bias=True, name=f'{name}_proj')(x)
    out = layers.Add(name=f'{name}_add')([x, y])
    out = layers.ReLU(name=f'{name}_relu2')(out)
    return out

def build_tcn_for_cubeai(L, C, num_classes):
    x_in = keras.Input(shape=(L, C), name='x')         # 단일 입력
    x = x_in
    for i, d in enumerate(DILS):
        x = tcn_block_bn(x, filters=FILTERS, k=KERNEL, d=d, name=f'tcn{i+1}', first_block=(i==0))
    x = layers.GlobalMaxPooling1D(name='gmp')(x)       # 마스크/람다 없이 패딩 무시
    x = layers.Dense(HEAD_UNITS, activation='relu', name='head')(x)
    out = layers.Dense(num_classes, activation='softmax', dtype='float32', name='pred')(x)  # 혼정에서도 FP32 출력
    return keras.Model(x_in, out, name='tcn_cubeai_ready')

model = build_tcn_for_cubeai(L, C, num_classes)
model.summary()

# -------------------- Loss: Focal + (optional) class_weight --------------------
classes, counts = np.unique(ytr, return_counts=True)
minor = classes[np.argmin(counts)]
alpha_vec = np.ones((num_classes,), dtype=np.float32) * 0.25
alpha_vec[minor] = 0.75
print(f"[INFO] focal alpha: {alpha_vec}, gamma=2.0")

def focal_loss(gamma=2.0, alpha=alpha_vec):
    sce = keras.losses.SparseCategoricalCrossentropy(reduction='none')
    alpha_c = tf.constant(alpha, dtype=tf.float32)
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0-1e-7)
        y_true = tf.cast(y_true, tf.int32)
        ce = sce(y_true, y_pred)                     # (B,)
        y_oh = tf.one_hot(y_true, depth=num_classes, dtype=tf.float32)
        p_t = tf.reduce_sum(y_oh * y_pred, axis=-1)  # (B,)
        a_t = tf.reduce_sum(y_oh * alpha_c, axis=-1) # (B,)
        fl = a_t * tf.pow(1.0 - p_t, gamma) * ce
        return tf.reduce_mean(fl)
    return loss

USE_CLASS_WEIGHT = True
class_weight = None
if USE_CLASS_WEIGHT:
    cw = compute_class_weight(class_weight='balanced', classes=np.arange(num_classes), y=ytr_os)
    class_weight = {int(i): float(w) for i, w in enumerate(cw)}
    print("[INFO] class_weight:", class_weight)

# -------------------- Callback: Macro-F1 on VAL + Cosine LR --------------------
class ValF1Callback(keras.callbacks.Callback):
    def __init__(self, ds_val, name='val_f1_macro', patience=8):
        super().__init__(); self.ds_val = ds_val; self.name=name
        self.best = -1.0; self.patience = patience; self.wait=0
    def on_epoch_end(self, epoch, logs=None):
        probs, y_true = [], []
        for xb, yb in self.ds_val:
            pb = self.model.predict(xb, verbose=0)
            probs.append(pb); y_true.append(yb.numpy())
        probs = np.concatenate(probs, axis=0)
        y_true = np.concatenate(y_true, axis=0)
        y_pred = probs.argmax(axis=1)
        f1m = f1_score(y_true, y_pred, average='macro', zero_division=0)
        logs = logs or {}
        logs[self.name] = f1m
        print(f"\n[VAL] macro-F1={f1m:.4f}")
        if f1m > self.best + 1e-6:
            self.best = f1m; self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"[EarlyStopping] No improvement in {self.patience} epochs on {self.name}.")
                self.model.stop_training = True

INIT_LR = 2.5e-3
EPOCHS  = 60
MAX_EPOCHS = EPOCHS

def cosine_decay(epoch, lr):
    import math
    t = min(epoch, MAX_EPOCHS)
    return 0.5 * INIT_LR * (1.0 + math.cos(math.pi * t / MAX_EPOCHS))

model.compile(optimizer=keras.optimizers.Adam(learning_rate=INIT_LR),
              loss=focal_loss(gamma=2.0, alpha=alpha_vec),
              metrics=['accuracy'])

valf1_cb = ValF1Callback(ds_va, name='val_f1_macro', patience=8)
lrs_cb   = keras.callbacks.LearningRateScheduler(cosine_decay, verbose=0)
ckpt_cb  = keras.callbacks.ModelCheckpoint(str(OUT_DIR / 'tcn_best.keras'),
                                           monitor='val_accuracy', save_best_only=True)

hist = model.fit(ds_tr,
                 validation_data=ds_va,
                 epochs=EPOCHS,
                 callbacks=[valf1_cb, lrs_cb, ckpt_cb],
                 class_weight=class_weight,
                 verbose=1)

# -------------------- Threshold tuning on VAL (binary only) --------------------
use_threshold = False
best_t = 0.5
if num_classes == 2:
    probs_va = []
    for xb, yb in ds_va:
        probs_va.append(model.predict(xb, verbose=0))
    probs_va = np.concatenate(probs_va, axis=0)
    p1_va = probs_va[:,1]
    best_f1 = -1
    for t in np.linspace(0.2, 0.8, 121):
        y_pred = (p1_va >= t).astype(int)
        f1m = f1_score(yva, y_pred, average='macro', zero_division=0)
        if f1m > best_f1:
            best_f1, best_t = f1m, t
    use_threshold = True
    print(f"[VAL] best threshold={best_t:.3f} | macro-F1={best_f1:.4f}")

# -------------------- Evaluation helpers --------------------
def eval_split(name, ds, y_true_full, use_thr=False, thr=0.5):
    probs = []
    for xb, yb in ds:
        probs.append(model.predict(xb, verbose=0))
    probs = np.concatenate(probs, axis=0)
    if (num_classes == 2) and use_thr:
        y_pred = (probs[:,1] >= thr).astype(int)
    else:
        y_pred = probs.argmax(axis=1)
    acc = (y_pred == y_true_full).mean()
    f1m = f1_score(y_true_full, y_pred, average='macro', zero_division=0)
    f1w = f1_score(y_true_full, y_pred, average='weighted', zero_division=0)
    prec= precision_score(y_true_full, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true_full, y_pred, average='macro', zero_division=0)
    print(f"\n[{name}] acc={acc:.4f} | F1(macro)={f1m:.4f} | F1(weighted)={f1w:.4f} | Precision={prec:.4f} | Recall={rec:.4f}")
    print("Classification report:")
    print(classification_report(y_true_full, y_pred, digits=4, zero_division=0))
    print("Confusion matrix:")
    print(confusion_matrix(y_true_full, y_pred))
    return dict(acc=acc, f1_macro=f1m, f1_weighted=f1w)

print("\n================ Final Evaluation ================")
_ = eval_split('VAL',  ds_va, yva, use_thr=use_threshold, thr=best_t)
_ = eval_split('TEST', ds_te, yte, use_thr=use_threshold, thr=best_t)

# -------------------- Export (FP32 clone → .h5) for STM32Cube.AI --------------------
mixed_precision.set_global_policy('float32')   # 내보낼 때는 FP32로
export_model = build_tcn_for_cubeai(L, C, num_classes)
export_model.set_weights([w.astype(np.float32) for w in model.get_weights()])
export_path = OUT_DIR / "tcn_cubeai_ready.h5"
export_model.save(str(export_path), include_optimizer=False)
print(f"\n[OK] Best training checkpoint: {OUT_DIR/'tcn_best.keras'}")
print(f"[OK] Exported FP32 model for Cube.AI: {export_path}")
# =====================================================================================


[INFO] NPZ: move_pad_fs50_L128_event_zero.npz
[INFO] Shapes -> Xtr:(616, 128, 6), Xva:(77, 128, 6), Xte:(78, 128, 6), classes=2
[INFO] Channels: base=6 -> used=8 (energy_added=True)
[INFO] Oversampled train: 616 -> 736 (class dist: [496 240])


Model: "tcn_cubeai_ready"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ x (InputLayer)      │ (None, 128, 8)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_c1 (Conv1D)    │ (None, 128, 48)   │      1,920 │ x[0][0]           │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_bn1            │ (None, 128, 48)   │        144 │ tcn1_c1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_relu1 (ReLU)   │ (None, 128, 48)   │          0 │ tcn1_bn1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_c2 (Conv1D)    │ (None, 128, 48)   │     11,520 │ tcn1_relu1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_proj (Conv1D)  │ (None, 128, 48)   │        432 │ x[0][0]           │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_bn2            │ (None, 128, 48)   │        192 │ tcn1_c2[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_add (Add)      │ (None, 128, 48)   │          0 │ tcn1_proj[0][0],  │
│                     │                   │            │ tcn1_bn2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_relu2 (ReLU)   │ (None, 128, 48)   │          0 │ tcn1_add[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_c1 (Conv1D)    │ (None, 128, 48)   │     11,520 │ tcn1_relu2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_bn1            │ (None, 128, 48)   │        192 │ tcn2_c1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_relu1 (ReLU)   │ (None, 128, 48)   │          0 │ tcn2_bn1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_c2 (Conv1D)    │ (None, 128, 48)   │     11,520 │ tcn2_relu1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_bn2            │ (None, 128, 48)   │        192 │ tcn2_c2[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_add (Add)      │ (None, 128, 48)   │          0 │ tcn1_relu2[0][0], │
│                     │                   │            │ tcn2_bn2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_relu2 (ReLU)   │ (None, 128, 48)   │          0 │ tcn2_add[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn3_c1 (Conv1D)    │ (None, 128, 48)   │     11,520 │ tcn2_relu2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn3_bn1            │ (None, 128, 48)   │        192 │ tcn3_c1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn3_relu1 (ReLU)   │ (None, 128, 48)   │          0 │ tcn3_bn1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn3_c2 (Conv1D)    │ (None, 128, 48)   │     11,520 │ tcn3_relu1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn3_bn2            │ (None, 128, 48)   │        192 │ tcn3_c2[0][0]     │
│ (BatchNormalizatio… │                   │            │                 

 Total params: 112,802 (440.63 KB)

 Trainable params: 111,842 (436.88 KB)

 Non-trainable params: 960 (3.75 KB)

[INFO] focal alpha: [0.25 0.75], gamma=2.0
[INFO] class_weight: {0: 0.7419354838709677, 1: 1.5333333333333334}
Epoch 1/60
21/23 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5593 - loss: 0.3602
[VAL] macro-F1=0.1881
23/23 ━━━━━━━━━━━━━━━━━━━━ 24s 292ms/step - accuracy: 0.5635 - loss: 0.3404 - val_accuracy: 0.2078 - val_loss: 0.6940 - val_f1_macro: 0.1881 - learning_rate: 0.0025
Epoch 2/60
21/23 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6615 - loss: 0.0788
[VAL] macro-F1=0.4697
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.6621 - loss: 0.0791 - val_accuracy: 0.4805 - val_loss: 0.2891 - val_f1_macro: 0.4697 - learning_rate: 0.0025
Epoch 3/60
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6947 - loss: 0.0809
[VAL] macro-F1=0.5253
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.6948 - loss: 0.0806 - val_accuracy: 0.5714 - val_loss: 0.1139 - val_f1_macro: 0.5253 - learning_rate: 0.0025
Epoch 4/60
21/23 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6921 - loss:


[TEST] acc=0.7179 | F1(macro)=0.5675 | F1(weighted)=0.7245 | Precision=0.5651 | Recall=0.5714
Classification report:
              precision    recall  f1-score   support

           0     0.8361    0.8095    0.8226        63
           1     0.2941    0.3333    0.3125        15

    accuracy                         0.7179        78
   macro avg     0.5651    0.5714    0.5675        78
weighted avg     0.7318    0.7179    0.7245        78

Confusion matrix:
[[51 12]
 [10  5]]

[OK] Best training checkpoint: /content/IMU_DATA_extracted/tcn_best.keras
[OK] Exported FP32 model for Cube.AI: /content/IMU_DATA_extracted/tcn_cubeai_ready.h5
